# Colab runner — RoBERTa seeds 2 and 3 (partitions 78516, 944601)

The first RoBERTa sweep covered only partition `5768`. The TF-IDF baseline used all three,
so this notebook runs the other two for RoBERTa (pooled + size control + all 25 LOBO folds
each) to put both models on equal footing and give the per-bank gaps real error bars.

**Runtime → Change runtime type → T4 GPU** before you start.

### Resuming
Results go to **Google Drive**, one JSON per fold. After any disconnect: reconnect, re-run
**Stages 0–3**, then re-run the sweep cell you were on. Finished folds are skipped, so a
disconnect costs at most one fold (~7 min).

### Budget
Each seed is 27 folds (1 pooled + 1 size control + 25 LOBO) at ~7 min ≈ **3.2 h per seed**,
~6.5 h for both. That is more than one free-tier session, so expect to reconnect once or
twice. The sweep is ordered so that seed 78516 is complete before 944601 starts.

Hyperparameters are **not** re-searched: `selected_hparams.json` from the first run is
reused (lr 1e-5, batch 32), so all three partitions share one model-selection budget.

## Stage 0 — confirm you have a GPU

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"
print("device:", torch.cuda.get_device_name(0))

## Stage 1 — mount Drive (same folder as the first run)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE_ROOT = '/content/drive/MyDrive/cs72_final_project'
os.environ['WCB_RESULTS_DIR'] = f'{DRIVE_ROOT}/results'
os.environ['WCB_FIGURES_DIR'] = f'{DRIVE_ROOT}/figures'
for p in (os.environ['WCB_RESULTS_DIR'], os.environ['WCB_FIGURES_DIR']):
    pathlib.Path(p).mkdir(parents=True, exist_ok=True)
print('results ->', os.environ['WCB_RESULTS_DIR'])

hp = pathlib.Path(os.environ['WCB_RESULTS_DIR'], 'roberta', 'selected_hparams.json')
assert hp.exists(), f"{hp} missing - the first RoBERTa run's results must be in this Drive folder"
print(hp.read_text()[:120])

## Stage 2 — get the code onto Colab

Same as the first run: unzip `final_project.zip` from the Drive folder. (If you have not
changed any code since, the existing zip is fine.)

In [ ]:
!rm -rf /content/final_project
!unzip -q {DRIVE_ROOT}/final_project.zip -d /content/
import os; os.chdir('/content/final_project')
!ls

## Stage 3 — install dependencies

Do **not** install torch (it would replace Colab's CUDA-matched build).

In [ ]:
!pip install -q datasets transformers nltk
!pip install -q -e . --no-deps
import nltk; nltk.download('punkt_tab', quiet=True)

import sys, os, pathlib
# A running kernel cannot see a package pip installed moments ago, and restarting would
# drop the Drive mount, so add src/ to sys.path directly.
root = next((p for p in ('/content/final_project', '/content/final_project/final_project')
             if pathlib.Path(p, 'src', 'wcb', 'config.py').exists()), None)
assert root, "couldn't find src/wcb - check Stage 2 actually unzipped"
os.chdir(root)
if f'{root}/src' not in sys.path:
    sys.path.insert(0, f'{root}/src')

from wcb import config as C
print('project :', root)
print('results :', C.RESULTS_DIR)
assert str(C.RESULTS_DIR).startswith('/content/drive'), 'results are NOT on Drive - re-run Stage 1'
print('\nOK')

## Stage 4 — how much is already done?

Safe to run at any time, including right after a disconnect.

In [ ]:
import glob, os
SEEDS = [5768, 78516, 944601]
print(f"{'seed':<8}{'pooled':>8}{'size_ctl':>10}{'lobo':>8}")
for s in SEEDS:
    n = {c: len(glob.glob(f"{C.RESULTS_DIR}/roberta/seed{s}/{c}/*.json"))
         for c in ('pooled', 'size_control', 'lobo')}
    print(f"{s:<8}{n['pooled']:>5}/25{n['size_control']:>7}/25{n['lobo']:>5}/25")

## Stage 5 — the sweep

Run **in order**. Every cell is resumable, so after a disconnect just re-run the one you
were on. `--no-save-pooled` skips the ~500 MB pooled checkpoint for these seeds: the demo
and the Czech check already use the seed-5768 one.

| cell | folds | time | what you have afterwards |
|---|---|---|---|
| 5a | 2 | ~15 min | seed 78516 ceiling + size control |
| 5b | 25 | ~3 h | seed 78516 complete |
| 5c | 2 | ~15 min | seed 944601 ceiling + size control |
| 5d | 25 | ~3 h | seed 944601 complete |

In [ ]:
# 5a - seed 78516: pooled + size control
!python scripts/run_roberta.py --seeds 78516 --condition pooled size_control --no-save-pooled

In [ ]:
# 5b - seed 78516: all 25 LOBO folds. Re-run after any disconnect.
!python scripts/run_roberta.py --seeds 78516 --condition lobo

In [ ]:
# 5c - seed 944601: pooled + size control
!python scripts/run_roberta.py --seeds 944601 --condition pooled size_control --no-save-pooled

In [ ]:
# 5d - seed 944601: all 25 LOBO folds. Re-run after any disconnect.
!python scripts/run_roberta.py --seeds 944601 --condition lobo

## Stage 6 — tables and figures across all seeds

Only run once Stage 4 shows every seed you want at 25/25. `make_report.py` averages over
whichever seeds it finds; pass `--seeds` to restrict it.

In [ ]:
!python scripts/make_report.py --model roberta

In [ ]:
import pandas as pd
b = pd.read_csv(f"{C.RESULTS_DIR}/roberta_gap_summary.csv")
print(b[['bank','pooled_weighted_f1','gap_data','gap_bank','gap_bank_std']].round(3)
      .sort_values('gap_bank', ascending=False).to_string(index=False))
print('\nmean gap_bank:', round(b.gap_bank.mean(), 4), '| mean gap_data:', round(b.gap_data.mean(), 4))
print(pd.read_csv(f"{C.RESULTS_DIR}/roberta_feature_correlations.csv").round(3).to_string(index=False))

In [ ]:
from IPython.display import Image, display
for f in sorted(glob.glob(f"{C.FIGURES_DIR}/roberta_fig*.png")):
    print(f); display(Image(f))